# XBRL validation (Part 11)

Loads the filing's own XBRL with Arelle, maps PDF row labels to concepts, and compares both extraction paths. Run `dvc repro xbrl` first; this notebook reads its outputs and re-runs one filing interactively.

In [1]:
import sys, pandas as pd
sys.path.insert(0, '../src')
import xbrl
from common import XBRL, load_manifest
pd.set_option('display.max_colwidth', 80)

## 1. Non-dimensional duration facts for the latest fiscal year (Exercise 11.1)

In [2]:
facts = pd.read_csv(XBRL / 'facts.csv', parse_dates=['end'])
tenk = [s for s in load_manifest() if '10K' in s][0]
f = facts[(facts.stem == tenk) & (~facts.instant) & (facts.dims.isna())]
latest = f.end.max()
f[(f.end == latest) & (f.days > 350)][['concept', 'label', 'value', 'decimals']].sort_values('concept')

,concept,label,value,decimals
379,AdjustmentsToAdditionalPaidInCapitalSharebasedCompensationRequisiteServicePe...,"APIC, Share-Based Payment Arrangement, Increase for Cost Recognition",506828000.0,-3.0
1015,AllocatedShareBasedCompensationExpense,"Share-Based Payment Arrangement, Expense",393378000.0,-3.0
519,AllowanceForDoubtfulAccountsReceivableRecoveries,"Accounts Receivable, Allowance for Credit Loss, Recovery",6901000.0,-3.0
857,AmortizationOfDebtIssuanceCostsIncludingCapitalizedInterest,"Amortization Of Debt Issuance Costs, Including Capitalized Interest",7802000.0,-3.0
180,AmortizationOfFinancingCosts,Amortization of Debt Issuance Costs,6521000.0,-3.0
...,...,...,...,...
1206,ValuationAllowanceDeferredTaxAssetChangeInAmount,"Valuation Allowance, Deferred Tax Asset, Increase (Decrease), Amount",-4100000.0,-5.0
1203,ValuationAllowanceRelease,"Valuation Allowance, Release",7558000.0,-3.0
886,VariableLeaseCost,"Variable Lease, Cost",93850000.0,-3.0
153,WeightedAverageNumberOfDilutedSharesOutstanding,"Weighted Average Number of Shares Outstanding, Diluted",154346000.0,-3.0


## 2. Label -> concept mapping and method per line

In [3]:
cmp = pd.read_csv(XBRL / 'comparison.csv')
cmp = cmp[cmp.status != 'no_table']
cmp.groupby(['path', 'statement', 'label', 'concept', 'method']).size().reset_index(name='periods')

,path,statement,label,concept,method,periods
0,docling,balance_sheet,accounts payable,AccountsPayableCurrent,manual,2
1,docling,balance_sheet,"accounts receivable, net of reserves of $3,522 and $3,469 at december 31, 20...",AccountsReceivableNetCurrent,manual,2
2,docling,balance_sheet,"accounts receivable, net of reserves of $5,769 and $3,522 at september 30, 2...",AccountsReceivableNetCurrent,manual,2
3,docling,balance_sheet,accrued expenses,AccruedLiabilitiesCurrent,manual,2
4,docling,balance_sheet,accumulated other comprehensive loss,AccumulatedOtherComprehensiveIncomeLossNetOfTax,manual,2
...,...,...,...,...,...,...
131,traditional,income_statement,restructuring (benefit) charge,RestructuringCharges,manual,4
132,traditional,income_statement,restructuring charge,RestructuringCharges,manual,3
133,traditional,income_statement,revenue,RevenueFromContractWithCustomerIncludingAssessedTax,manual,7
134,traditional,income_statement,sales and marketing,SellingAndMarketingExpense,manual,7


## 3. Match rate per statement and per extraction path

In [4]:
cmp.groupby(['stem', 'statement', 'path']).status.apply(lambda s: (s == 'match').mean()).unstack('path')

path                                 docling  traditional
stem              statement                              
AKAM_10K_20241231 balance_sheet     0.985294     0.955882
                  income_statement  0.850000     0.850000
AKAM_10Q_20250930 balance_sheet     1.000000     0.837209
                  income_statement  0.894737     0.894737

## 4. Every non-match with its suggested cause

Confirm each cause (OCR, table structure, normalization, mapping, period alignment, extension concept, rounding) and record the fix in `reports/xbrl.md`.

In [5]:
cmp[cmp.status != 'match'][['stem', 'path', 'statement', 'label', 'period', 'raw', 'pdf_value', 'concept', 'xbrl_value', 'status', 'suggested_cause']]

,stem,path,statement,label,period,raw,pdf_value,concept,xbrl_value,status,suggested_cause
30,AKAM_10K_20241231,traditional,income_statement,interest expense,2024,"(27,117)",-2.711700e+07,InterestExpenseNonoperating,2.711700e+07,sign,presentation: statement shows the value negated vs XBRL (sign policy)
31,AKAM_10K_20241231,traditional,income_statement,interest expense,2023,"(17,709)",-1.770900e+07,InterestExpenseNonoperating,1.770900e+07,sign,presentation: statement shows the value negated vs XBRL (sign policy)
32,AKAM_10K_20241231,traditional,income_statement,interest expense,2022,"(11,096)",-1.109600e+07,InterestExpenseNonoperating,1.109600e+07,sign,presentation: statement shows the value negated vs XBRL (sign policy)
33,AKAM_10K_20241231,traditional,income_statement,"other expense, net",2024,"(19,561)",-1.956100e+07,OtherNonoperatingExpenseNet,1.956100e+07,sign,presentation: statement shows the value negated vs XBRL (sign policy)
34,AKAM_10K_20241231,traditional,income_statement,"other expense, net",2023,"(12,296)",-1.229600e+07,OtherNonoperatingExpenseNet,1.229600e+07,sign,presentation: statement shows the value negated vs XBRL (sign policy)
35,AKAM_10K_20241231,traditional,income_statement,"other expense, net",2022,"(10,433)",-1.043300e+07,OtherNonoperatingExpenseNet,1.043300e+07,sign,presentation: statement shows the value negated vs XBRL (sign policy)
39,AKAM_10K_20241231,traditional,income_statement,provision for income taxes,2024,"(82,095)",-8.209500e+07,IncomeTaxExpenseBenefit,8.209500e+07,sign,presentation: statement shows the value negated vs XBRL (sign policy)
40,AKAM_10K_20241231,traditional,income_statement,provision for income taxes,2023,"(106,373)",-1.063730e+08,IncomeTaxExpenseBenefit,1.063730e+08,sign,presentation: statement shows the value negated vs XBRL (sign policy)
41,AKAM_10K_20241231,traditional,income_statement,provision for income taxes,2022,"(126,696)",-1.266960e+08,IncomeTaxExpenseBenefit,1.266960e+08,sign,presentation: statement shows the value negated vs XBRL (sign policy)
114,AKAM_10K_20241231,traditional,balance_sheet,"at december 31, 2023",2024,"1,556",1.556000e+06,NaN,NaN,xbrl_missing,"mapping: extension, dimensional or unmapped concept"


## 5. Re-run one filing interactively (after changing `config/label_map.yaml`)

In [6]:
m = load_manifest()[tenk]
fx, lab2con = xbrl.load_facts(xbrl.ixbrl_file(m['accession']), tenk)
redo = pd.DataFrame(xbrl.validate(tenk, m, fx, xbrl.load_map(), lab2con))
redo[redo.status != 'no_table'].groupby(['path', 'statement']).status.value_counts().unstack(fill_value=0)

2026-10-03 17:51:11,025 [ix11.11.1.2:invalidTransformation] Fact ecd:TrdArrDuration has unrecognized transformation namespace http://www.sec.gov/inlineXBRL/transformation/2015-08-31 - akam-20241231.htm 8

2026-10-03 17:51:11,025 [ix11.11.1.2:invalidTransformation] Fact ecd:TrdArrDuration has unrecognized transformation namespace http://www.sec.gov/inlineXBRL/transformation/2015-08-31 - akam-20241231.htm 8

2026-10-03 17:51:11,026 [ix11.11.1.2:invalidTransformation] Fact dei:DocumentAnnualReport has unrecognized transformation namespace http://www.sec.gov/inlineXBRL/transformation/2015-08-31 - akam-20241231.htm 8

2026-10-03 17:51:11,026 [ix11.11.1.2:invalidTransformation] Fact dei:DocumentTransitionReport has unrecognized transformation namespace http://www.sec.gov/inlineXBRL/transformation/2015-08-31 - akam-20241231.htm 8

2026-10-03 17:51:11,026 [ix11.11.1.2:invalidTransformation] Fact dei:EntityIncorporationStateCountryCode has unrecognized transformation namespace http://www.sec.go

status                        match  sign  xbrl_missing
path        statement                                  
docling     balance_sheet        67     1             0
            income_statement     51     9             0
traditional balance_sheet        65     1             2
            income_statement     51     9             0